# Predicting EV Purchases: Exploratory Data Analysis

Notebook 1 of 3 for the Kaggle competition Playground Series S6E9. The task is to predict whether a person will buy an electric vehicle (`Will_Buy_EV`), and submissions are scored by the ROC AUC of the predicted probabilities. This notebook looks at the data before any modelling: the numeric columns, the purchase rate in each category, the correlations, and the repeated income values that the target encoding in notebook 2 is built on.

It reads the competition files `train.csv`, `test.csv` and `sample_submission.csv` (licensed CC BY 4.0) and writes four figures to `figures/`. Notebook 2 builds the features and trains the models, and notebook 3 stacks them with public out-of-fold predictions and writes the submission; neither reads anything from this notebook.

## Setup

Import the libraries and set the random seed. `SMOKE` samples a small part of the data for a quick check run on Kaggle; it is off in this version.

In [ ]:
import glob
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SMOKE = False
SEED = 42
FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)
print(f"smoke run: {SMOKE}")
print(f"pandas {pd.__version__}, numpy {np.__version__}, matplotlib {matplotlib.__version__}, seaborn {sns.__version__}")

**Conclusion.** The smoke switch is off, so this is the full run. The Kaggle image provides pandas 2.3.3, NumPy 2.1.3, Matplotlib 3.10.0 and seaborn 0.13.2, and nothing has to be installed.

## Load the data

Read the train, test and sample submission files and turn the target into 0 and 1. The five rows shown come from the competition's `train.csv` (Kaggle Playground Series S6E9, CC BY 4.0).

In [ ]:
root = glob.glob("/kaggle/input/**/train.csv", recursive=True)[0].rsplit("/", 1)[0]
train = pd.read_csv(f"{root}/train.csv")
test = pd.read_csv(f"{root}/test.csv")
sample = pd.read_csv(f"{root}/sample_submission.csv")
if SMOKE:
    train = train.sample(30_000, random_state=SEED).reset_index(drop=True)
    test = test.sample(10_000, random_state=SEED).reset_index(drop=True)

TARGET = "Will_Buy_EV"
y = (train[TARGET] == "Yes").astype(int).to_numpy()
print(train.shape, test.shape)
print(train.isna().sum().sum(), test.isna().sum().sum())
train.head()

**Conclusion.** The train set has 668,665 rows and 15 columns (an id, 13 attributes and the target), and the test set has 286,571 rows without the target. Neither file has a missing value, so no imputation is needed.

## Numeric columns

Summarise every numeric column and plot its distribution split by the target. If the two distributions differ a lot, the column carries signal. The mean of each column for non-buyers and buyers is printed so the differences can be read as numbers.

In [ ]:
numeric = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
           "Charging_Stations_Near_Home", "Charging_Stations_Near_Work", "Environmental_Concern_Level"]
categorical = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible",
               "Subsidy_Available", "Range_Anxiety_Level"]

print(f"positive rate: {y.mean():.4f}")
display(train[numeric].describe().T)
print(train[numeric].groupby(y).mean().T.rename(columns={0: "non-buyers", 1: "buyers"}).round(3).to_string())
print(f"share of rows at the 30000 income floor: {(train.Annual_Income_USD == 30000).mean():.3f}")

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, col in zip(axes.ravel(), numeric):
    sns.histplot(data=train, x=col, hue=TARGET, bins=40, stat="density", common_norm=False, element="step", ax=ax)
    ax.set_title(col)
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.savefig(FIGURES / "01_numeric_distributions.png", dpi=110)
plt.show()

**Conclusion.** 17.5% of the people in the train set buy an electric vehicle, so the classes are unbalanced and a ranking metric such as AUC suits the task. Two columns separate buyers from non-buyers clearly: buyers earn 98,827 on average against 81,795, and their environmental concern averages 4.38 against 2.63 on the 1 to 5 scale. Age (46.8 against 47.1), the number of cars (1.72 against 1.71) and the charging stations near home and work barely differ. The income histogram has a tall spike at exactly 30,000 that holds 9.2% of all rows, which looks like a lower limit applied when the data was made, and the commute histogram has a similar spike at its lowest value.

## Purchase rate by category

Show the purchase rate for each level of every categorical column, plus environmental concern and the number of cars owned. The dashed red line is the overall purchase rate. The rates are printed as well as plotted.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, col in zip(axes.ravel(), categorical + ["Environmental_Concern_Level", "Number_of_Cars_Owned"]):
    rate = train.assign(y=y).groupby(col)["y"].mean().sort_values()
    print(f"{col}: " + ", ".join(f"{level}={value:.3f}" for level, value in rate.items()))
    sns.barplot(x=rate.index.astype(str), y=rate.values, ax=ax, color="steelblue")
    ax.axhline(y.mean(), color="crimson", ls="--")
    ax.set_title(f"P(buy) by {col}")
    ax.set_ylabel("")
plt.tight_layout()
plt.savefig(FIGURES / "02_purchase_rate_by_category.png", dpi=110)
plt.show()

**Conclusion.** Three columns drive the purchase rate. It rises from 0.6% at environmental concern 1 to 51.8% at concern 5; it is 0.6% without a subsidy and 27.5% with one; and it falls from 18.9% at low range anxiety to 4.2% at medium and 0.1% at high. Home charging matters less (12.7% without, 19.6% with), and city type (16.1% urban to 19.3% rural) and the current car type (15.6% for trucks to 18.1% for SUVs) a little. Gender (17.2% to 17.8%) and the number of cars (16.6% to 17.8%) hardly change it. Since almost nobody buys without a subsidy or with high range anxiety, the decision looks like a threshold on a few columns rather than a smooth mix of all of them.

## Correlations

Show how the numeric columns and the encoded yes/no columns relate to each other and to the target. The cell prints every column's correlation with the target and the five strongest correlations between two inputs.

In [ ]:
encoded = train[numeric].assign(
    Subsidy=(train.Subsidy_Available == "Yes").astype(int),
    HomeCharging=(train.Home_Charging_Possible == "Yes").astype(int),
    RangeAnxiety=train.Range_Anxiety_Level.map({"Low": 0, "Medium": 1, "High": 2}),
    Target=y,
)
corr = encoded.corr()
inputs = corr.drop(index="Target", columns="Target")
pairs = inputs.where(np.triu(np.ones(inputs.shape, dtype=bool), k=1)).stack()
print(corr["Target"].drop("Target").sort_values().round(3).to_string())
print(pairs.reindex(pairs.abs().sort_values(ascending=False).index).head(5).round(3).to_string())
plt.figure(figsize=(10, 8))
sns.heatmap(encoded.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.savefig(FIGURES / "03_correlations.png", dpi=110, bbox_inches="tight")
plt.show()

**Conclusion.** The target's strongest linear links are environmental concern (0.464), subsidy (0.342) and income (0.226); range anxiety is negative (-0.115) and home charging is small (0.084). Every other column is within 0.05 of zero. Among the inputs, the stations near home and near work go together (0.511), people who can charge at home report less range anxiety (-0.471) and have fewer stations nearby (-0.398 near home, -0.384 near work), and a longer commute comes with more anxiety (0.191). A small set of columns carries most of the signal, so the modelling is about representing those columns well.

## Repeated income values

Income values repeat a lot. This cell counts how often each exact income appears and plots the purchase rate of every income value above the floor that is seen at least 50 times.

In [ ]:
income_counts = train.Annual_Income_USD.value_counts()
print(f"share of rows at the 30000 floor: {(train.Annual_Income_USD == 30000).mean():.3f}")
print(f"distinct incomes: {income_counts.size}, share of rows whose income appears 20+ times: {income_counts[income_counts >= 20].sum() / len(train):.3f}")

by_income = train.assign(y=y).groupby("Annual_Income_USD")["y"].agg(["mean", "size"])
by_income = by_income[(by_income["size"] >= 50) & (by_income.index > 30000)]
print(f"income values above the floor with 50+ rows: {len(by_income)}, purchase rate from {by_income['mean'].min():.3f} to {by_income['mean'].max():.3f}")
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
ax[0].scatter(by_income.index, by_income["mean"], s=8, alpha=0.6)
ax[0].set(xlabel="Annual_Income_USD", ylabel="P(buy)", title="Purchase rate per exact income value")
sns.histplot(income_counts.clip(upper=200), bins=50, ax=ax[1])
ax[1].set(title="Rows per exact income value")
plt.savefig(FIGURES / "04_income_values.png", dpi=110, bbox_inches="tight")
plt.show()

**Conclusion.** Only 13,214 different incomes cover 668,665 rows, and 96.5% of the rows have an income that appears at least 20 times. Among the 4,017 incomes above the floor with at least 50 rows, the purchase rate runs from 0% to 100%: it rises with income, as the left panel shows, but jumps a lot from one exact value to the next. A model can learn the rate of each exact value, which is what the target encoding in notebook 2 does, inside each fold so that no row sees its own label.

## Summary

The purchase decision depends mostly on four columns: environmental concern (0.6% to 51.8% across its five levels), the subsidy (0.6% without, 27.5% with), range anxiety (18.9% at low, 0.1% at high) and income (buyers average 98,827 against 81,795). Notebook 2 turns these into a recipe score that is both a feature and the starting point of the boosted models.

Incomes repeat heavily, with 13,214 distinct values for 668,665 rows, so notebook 2 also encodes each exact income by its purchase rate, built inside each fold.